In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
base_path = dbutils.widgets.get("base_path")

bronze_table_path = f"{base_path}/bronze/earthquakes_bronze"
silver_scd2_path = f"{base_path}/silver/earthquakes_scd2"


In [0]:
# 1. Read and deduplicate Bronze stream
df_bronze = spark.read.format("delta").load(bronze_table_path)
window_spec = Window.partitionBy("event_id").orderBy(F.col("event_time").desc())

df_stage = (
    df_bronze
    .filter(F.col("_rescued_data").isNull() & F.col("event_id").isNotNull())
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num", "_rescued_data")
)

# 2. Initialize Silver SCD2 table if non-existent
if not DeltaTable.isDeltaTable(spark, silver_scd2_path):
    (
        df_stage
        .withColumn("valid_from", F.current_timestamp())
        .withColumn("valid_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .write.format("delta")
        .mode("overwrite")
        .save(silver_scd2_path)
    )
else:
    # 3. SCD Type 2 Single-Pass MERGE
    target_table = DeltaTable.forPath(spark, silver_scd2_path)
    df_target_active = target_table.toDF().filter("is_current = true")

    # Detect records with attribute changes against current active target rows
    df_changes = (
        df_stage.alias("stg")
        .join(df_target_active.alias("tgt"), "event_id", "inner")
        .filter(
            (F.col("stg.magnitude") != F.col("tgt.magnitude")) |
            (F.col("stg.depth_km") != F.col("tgt.depth_km")) |
            (F.col("stg.event_time") != F.col("tgt.event_time"))
        )
        .select("stg.*")
    )

    # Union dataset: 
    # - Non-null merge_key matches active rows to expire them (UPDATE)
    # - Null merge_key forces INSERT for new keys and new historical versions
    staged_updates = (
        df_changes.withColumn("merge_key", F.col("event_id"))
        .unionByName(
            df_stage.withColumn("merge_key", F.lit(None).cast("string"))
        )
    )

    (
        target_table.alias("tgt")
        .merge(
            staged_updates.alias("stg"),
            "tgt.event_id = stg.merge_key AND tgt.is_current = true"
        )
        .whenMatchedUpdate(
            condition="stg.merge_key IS NOT NULL",
            set={
                "is_current": "false",
                "valid_to": "current_timestamp()"
            }
        )
        .whenNotMatchedInsert(
            values={
                "event_id": "stg.event_id",
                "title": "stg.title",
                "magnitude": "stg.magnitude",
                "place": "stg.place",
                "event_time": "stg.event_time",
                "longitude": "stg.longitude",
                "latitude": "stg.latitude",
                "depth_km": "stg.depth_km",
                "tsunami_flag": "stg.tsunami_flag",
                "_ingested_at": "stg._ingested_at",
                "_source_file": "stg._source_file",
                "valid_from": "current_timestamp()",
                "valid_to": "NULL",
                "is_current": "true"
            }
        )
        .execute()
    )

In [0]:
# current_user = spark.sql("SELECT current_user()").collect()[0][0]
# default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

# dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
# base_path = dbutils.widgets.get("base_path")

# silver_scd2_path = f"{base_path}/silver/earthquakes_scd2"
# df_silver_scd2 = spark.read.format("delta").load(silver_scd2_path)

# print("=== SILVER SCD TYPE 2 VERIFICATION ===")
# print(f"Total Historical Rows: {df_silver_scd2.count()}")
# print(f"Active Current Rows: {df_silver_scd2.filter('is_current = true').count()}")
# print(f"Expired Historical Rows: {df_silver_scd2.filter('is_current = false').count()}")

# df_silver_scd2.select(
#     "event_id", "magnitude", "depth_km", "valid_from", "valid_to", "is_current"
# ).show(10, truncate=False)